# Sign and Basis Invariant Networks for Spectral Graph Learning

> Lim, Robinson, Zhao et al. (2022). *Sign and Basis Invariant Networks for Spectral Graph Representation Learning*. arXiv 2202.13013.

## Learning Objectives

1. **Explain** the sign and basis ambiguity in Laplacian eigenvectors and why it breaks naive positional encodings
2. **Derive** the sign invariance condition: $h(v) = \phi(v) + \phi(-v)$ (Proposition 1)
3. **Derive** the basis invariance condition via the projection $V \mapsto VV^\top$ (Proposition 2)
4. **Implement** SignNet positional encodings and verify sign invariance in code
5. **Implement** the simplified BasisNet encoding and verify $O(d)$ invariance
6. **State** how SignNet/BasisNet strictly generalise spectral graph convolutions and prior positional encodings

## The Eigenvector Ambiguity Problem

### Laplacian Eigenvectors

For a graph $G=(V,E)$ with adjacency matrix $A$ and degree matrix $D$, the **normalised Laplacian** is:
$$L = I - D^{-1/2}AD^{-1/2}$$

$L$ is symmetric PSD, so its eigendecomposition is $L = V\Lambda V^\top$ with real eigenvalues $\lambda_1 \le \lambda_2 \le \cdots \le \lambda_n$ and orthonormal eigenvectors $v_1,\ldots,v_n$.

Eigenvectors encode structural information: the **Fiedler vector** $v_2$ partitions nodes by community, higher eigenvectors capture finer structure, and the eigenvalues quantify frequency.

### Sign Ambiguity

If $v$ is an eigenvector of $L$ for eigenvalue $\lambda$, then so is $-v$:
$$L(-v) = -(Lv) = -(\lambda v) = \lambda(-v)$$

For $k$ eigenvectors there are $2^k$ valid sign combinations. A naive positional encoding $\text{PE}(u) = [v_{1,u},\ldots,v_{k,u}]$ will produce **different outputs for the same graph** depending on which sign convention the eigensolver chooses — this is catastrophic for learning.

**Sign invariance** requirement: a function $f : \mathbb{R}^{n\times k} \to \mathbb{R}^{d_{\text{out}}}$ must satisfy
$$f(v_1,\ldots,v_k) = f(s_1 v_1,\ldots,s_k v_k), \quad s_i \in \{-1,+1\} \tag{1}$$

### Basis Ambiguity

When eigenvalue $\lambda$ has multiplicity $d > 1$, its eigenspace is $d$-dimensional. Any orthonormal basis $V \in \mathbb{R}^{n \times d}$ of that eigenspace is valid, and so is $VQ$ for any orthogonal $Q \in O(d)$. With $k$ eigenvectors there are infinitely many valid basis choices.

**Basis invariance** requirement:
$$f(V_1,\ldots,V_l) = f(V_1 Q_1,\ldots,V_l Q_l), \quad Q_i \in O(d_i) \tag{2}$$

Sign invariance is the special case $O(1) = \{-1, +1\}$.

**Why this matters in practice**: 64% of molecule graphs in the ZINC benchmark have at least one degenerate eigenspace, so basis ambiguity is not a theoretical edge case.

In [1]:
from IPython.display import SVG, display
svg = '''
<svg xmlns="http://www.w3.org/2000/svg" width="860" height="360" font-family="monospace" font-size="11">
  <rect width="860" height="360" fill="#fafafa" rx="8"/>
  <text x="430" y="22" text-anchor="middle" fill="#333" font-size="13" font-weight="bold">Eigenvector Ambiguities and the SignNet/BasisNet Solution</text>

  <!-- Left panel: sign ambiguity -->
  <rect x="10" y="32" width="260" height="316" rx="5" fill="#fce8e8" stroke="#e15759"/>
  <text x="140" y="50" text-anchor="middle" fill="#e15759" font-weight="bold">Sign Ambiguity</text>
  <text x="20" y="68" fill="#555" font-size="9">Both v and -v satisfy Lv = \u03bbv.</text>
  <!-- path graph -->
  <circle cx="50" cy="105" r="13" fill="#4e79a7"/><text x="50" y="109" text-anchor="middle" fill="white" font-size="9">0</text>
  <circle cx="100" cy="105" r="13" fill="#4e79a7"/><text x="100" y="109" text-anchor="middle" fill="white" font-size="9">1</text>
  <circle cx="150" cy="105" r="13" fill="#4e79a7"/><text x="150" y="109" text-anchor="middle" fill="white" font-size="9">2</text>
  <circle cx="200" cy="105" r="13" fill="#4e79a7"/><text x="200" y="109" text-anchor="middle" fill="white" font-size="9">3</text>
  <line x1="63" y1="105" x2="87" y2="105" stroke="#333" stroke-width="1.5"/>
  <line x1="113" y1="105" x2="137" y2="105" stroke="#333" stroke-width="1.5"/>
  <line x1="163" y1="105" x2="187" y2="105" stroke="#333" stroke-width="1.5"/>
  <!-- eigenvec v -->
  <text x="20" y="135" fill="#333" font-size="9" font-weight="bold">v\u2082 (one valid eigenvec):</text>
  <line x1="50" y1="160" x2="50" y2="148" stroke="#59a14f" stroke-width="2"/>
  <line x1="100" y1="160" x2="100" y2="143" stroke="#59a14f" stroke-width="2"/>
  <line x1="150" y1="160" x2="150" y2="177" stroke="#e15759" stroke-width="2"/>
  <line x1="200" y1="160" x2="200" y2="172" stroke="#e15759" stroke-width="2"/>
  <line x1="30" y1="160" x2="220" y2="160" stroke="#aaa" stroke-width="0.8"/>
  <text x="42" y="145" fill="#59a14f" font-size="8">+</text><text x="92" y="140" fill="#59a14f" font-size="8">+</text>
  <text x="142" y="183" fill="#e15759" font-size="8">-</text><text x="192" y="178" fill="#e15759" font-size="8">-</text>
  <!-- eigenvec -v -->
  <text x="20" y="200" fill="#333" font-size="9" font-weight="bold">-v\u2082 (equally valid):</text>
  <line x1="50" y1="225" x2="50" y2="237" stroke="#e15759" stroke-width="2"/>
  <line x1="100" y1="225" x2="100" y2="242" stroke="#e15759" stroke-width="2"/>
  <line x1="150" y1="225" x2="150" y2="208" stroke="#59a14f" stroke-width="2"/>
  <line x1="200" y1="225" x2="200" y2="213" stroke="#59a14f" stroke-width="2"/>
  <line x1="30" y1="225" x2="220" y2="225" stroke="#aaa" stroke-width="0.8"/>
  <text x="42" y="242" fill="#e15759" font-size="8">-</text><text x="92" y="247" fill="#e15759" font-size="8">-</text>
  <text x="142" y="205" fill="#59a14f" font-size="8">+</text><text x="192" y="210" fill="#59a14f" font-size="8">+</text>
  <text x="20" y="272" fill="#e15759" font-size="9">Naive LapPE gives DIFFERENT output</text>
  <text x="20" y="285" fill="#e15759" font-size="9">for the same graph \u2192 learning fails.</text>
  <text x="20" y="305" fill="#333" font-size="9">k eigenvecs \u2192 2\u1d4f sign combinations.</text>
  <text x="20" y="320" fill="#333" font-size="9">For k=10: 1024 possible PE outputs!</text>

  <!-- Middle panel: BasisNet/SignNet formulas -->
  <rect x="285" y="32" width="280" height="316" rx="5" fill="#e8f0fb" stroke="#4e79a7"/>
  <text x="425" y="50" text-anchor="middle" fill="#4e79a7" font-weight="bold">Invariant Architectures</text>

  <text x="295" y="70" fill="#333" font-size="9" font-weight="bold">Proposition 1 (sign invariance):</text>
  <text x="295" y="84" fill="#555" font-size="9">h: \u211d\u207f \u2192 \u211d is sign-invariant iff</text>
  <text x="295" y="98" fill="#4e79a7" font-size="10">h(v) = \u03c6(v) + \u03c6(-v)</text>
  <text x="295" y="112" fill="#555" font-size="9">for a continuous \u03c6: \u211d\u207f \u2192 \u211d.</text>

  <line x1="295" y1="122" x2="555" y2="122" stroke="#ddd"/>
  <text x="295" y="137" fill="#333" font-size="9" font-weight="bold">SignNet (Eq. 5):</text>
  <text x="295" y="151" fill="#4e79a7" font-size="9">f(v\u2081,...,v\u2096) = \u03c1( [\u03c6(v\u1d62)+\u03c6(-v\u1d62)]\u1d62 )</text>
  <text x="295" y="165" fill="#555" font-size="9">\u03c6, \u03c1: permutation-equivariant nets (MLP/GIN)</text>
  <text x="295" y="179" fill="#555" font-size="9">\u03c6(v)+\u03c6(-v) \u2192 sign invariant per eigenvec</text>

  <line x1="295" y1="192" x2="555" y2="192" stroke="#ddd"/>
  <text x="295" y="207" fill="#333" font-size="9" font-weight="bold">Proposition 2 (basis invariance):</text>
  <text x="295" y="221" fill="#555" font-size="9">V \u21a6 VV\u1d40 is O(d) invariant: (VQ)(VQ)\u1d40 = VV\u1d40</text>

  <line x1="295" y1="234" x2="555" y2="234" stroke="#ddd"/>
  <text x="295" y="249" fill="#333" font-size="9" font-weight="bold">BasisNet (Eq. 7):</text>
  <text x="295" y="263" fill="#4e79a7" font-size="9">f(V\u2081,...,V\u2097) = \u03c1( [IGN(V\u1d62V\u1d62\u1d40)]\u1d62 )</text>
  <text x="295" y="277" fill="#555" font-size="9">IGN: invariant graph network (Maron+ 2018)</text>
  <text x="295" y="291" fill="#555" font-size="9">maps n\u00d7n \u2192 n, permutation equivariant</text>

  <line x1="295" y1="305" x2="555" y2="305" stroke="#ddd"/>
  <text x="295" y="320" fill="#555" font-size="9">SignNet \u2282 BasisNet (sign = 1D basis change)</text>
  <text x="295" y="334" fill="#555" font-size="9">Both \u2282 Expressive-BasisNet (higher-order IGN)</text>

  <!-- Right panel: pipeline + theoretical power -->
  <rect x="580" y="32" width="270" height="316" rx="5" fill="#e8f8e8" stroke="#59a14f"/>
  <text x="715" y="50" text-anchor="middle" fill="#59a14f" font-weight="bold">Theoretical Power</text>

  <!-- pipeline boxes -->
  <rect x="590" y="58" width="100" height="28" rx="4" fill="#4e79a7" opacity="0.8"/>
  <text x="640" y="76" text-anchor="middle" fill="white" font-size="9">Graph (A, X)</text>
  <rect x="590" y="96" width="100" height="28" rx="4" fill="#f28e2b" opacity="0.8"/>
  <text x="640" y="114" text-anchor="middle" fill="white" font-size="9">Laplacian eigvecs</text>
  <rect x="590" y="134" width="100" height="28" rx="4" fill="#e15759" opacity="0.8"/>
  <text x="640" y="152" text-anchor="middle" fill="white" font-size="9">SignNet / BasisNet</text>
  <rect x="590" y="172" width="100" height="28" rx="4" fill="#59a14f" opacity="0.8"/>
  <text x="640" y="190" text-anchor="middle" fill="white" font-size="9">GNN / Transformer</text>
  <line x1="640" y1="86" x2="640" y2="96" stroke="#555" stroke-width="1.5" marker-end="url(#arr)"/>
  <line x1="640" y1="124" x2="640" y2="134" stroke="#555" stroke-width="1.5"/>
  <line x1="640" y1="162" x2="640" y2="172" stroke="#555" stroke-width="1.5"/>
  <text x="705" y="114" fill="#555" font-size="8">\u03c6(v)+\u03c6(-v)</text>
  <text x="705" y="128" fill="#555" font-size="8">\u2192 invariant PE</text>

  <line x1="580" y1="210" x2="840" y2="210" stroke="#ddd"/>
  <text x="590" y="225" fill="#333" font-size="9" font-weight="bold">Theorem 2:</text>
  <text x="590" y="239" fill="#555" font-size="9">SignNet universally approximates</text>
  <text x="590" y="253" fill="#555" font-size="9">all spectral graph convolutions:</text>
  <text x="590" y="267" fill="#4e79a7" font-size="9">f = V Diag(\u03b8) V\u1d40 X</text>

  <line x1="580" y1="278" x2="840" y2="278" stroke="#ddd"/>
  <text x="590" y="293" fill="#333" font-size="9" font-weight="bold">Prop. 4 (subsumes prior PEs):</text>
  <text x="590" y="307" fill="#555" font-size="9">\u2022 Heat kernel PE (diagonal V e^{-t\u039b} V\u1d40)</text>
  <text x="590" y="321" fill="#555" font-size="9">\u2022 Random walk PE (diag V \u039b\u207f V\u1d40)</text>
  <text x="590" y="335" fill="#555" font-size="9">\u2022 Generalized PageRank PE</text>
</svg>
'''
display(SVG(svg))

## Laplacian Eigenvectors and Sign Ambiguity

The normalized Laplacian is $L = I - D^{-1/2}AD^{-1/2}$. Its eigendecomposition satisfies:
- $\lambda_1 = 0$ always (constant eigenvector $\mathbf{1}/\sqrt{n}$)
- $\lambda_2 > 0$ iff the graph is connected (Fiedler value)
- All $\lambda_i \in [0, 2]$

For a **path graph** $0-1-2-\cdots-(n{-}1)$, the eigenvectors are sinusoidal functions of node index — the graph analogue of Fourier modes. Any numeric solver can return $v$ or $-v$ with equal validity.

In [2]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


def normalized_laplacian(A):
    """L = I - D^{-1/2} A D^{-1/2}"""
    d = A.sum(1)
    d_inv_sqrt = np.where(d > 0, 1.0 / np.sqrt(d), 0.0)
    D_inv_sqrt = np.diag(d_inv_sqrt)
    return np.eye(len(A)) - D_inv_sqrt @ A @ D_inv_sqrt


# Path graph 0-1-2-3-4-5
n = 6
A = np.zeros((n, n))
for i in range(n - 1):
    A[i, i + 1] = A[i + 1, i] = 1.0

L = normalized_laplacian(A)
vals, vecs = np.linalg.eigh(L)

print("Eigenvalues (\u03bb\u2081 = 0 = trivial constant mode):")
print(" ", vals.round(4))
print()
print("Eigenvectors (columns correspond to eigenvalues above):")
print(vecs.round(3))

# ── Demonstrate sign ambiguity ─────────────────────────────────────────────
v2 = vecs[:, 1]
print("\nv\u2082 is a valid eigenvector:",
      np.allclose(L @ v2, vals[1] * v2))
print("-v\u2082 is equally valid:     ",
      np.allclose(L @ (-v2), vals[1] * (-v2)))

print()
print("Naive LapPE for node 0 using v\u2082:   ", round(v2[0], 4))
print("""Naive LapPE for node 0 using -v\u2082:  """, round(-v2[0], 4))
print("\u2192 Same graph, same node, DIFFERENT positional encoding!")

# ── Plot eigenvectors ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
colors = ['#4e79a7', '#e15759', '#59a14f']
for k, ax in enumerate(axes):
    v = vecs[:, k + 1]
    ax.bar(range(n), v, color=colors[k], alpha=0.8, width=0.6, label=f'v{chr(0x2080 + k+2)}')
    ax.bar(range(n), -v, color=colors[k], alpha=0.3, width=0.6, hatch='//',
           label=f'-v{chr(0x2080 + k+2)} (equally valid)')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'Eigenvector v{chr(0x2080 + k+2)}  (\u03bb = {vals[k+1]:.3f})')
    ax.set_xlabel('Node index')
    ax.legend(fontsize=7)
    ax.set_ylim(-0.7, 0.7)
plt.suptitle('Path graph: both v and -v are valid eigenvectors (sign ambiguity)', fontsize=10)
plt.tight_layout()
plt.savefig('laplacian_eigenvectors.png', dpi=110)
plt.show()
print('Saved laplacian_eigenvectors.png')

Eigenvalues (λ₁ = 0 = trivial constant mode):
  [0.    0.191 0.691 1.309 1.809 2.   ]

Eigenvectors (columns correspond to eigenvalues above):
[[ 0.316  0.447 -0.447  0.447 -0.447 -0.316]
 [ 0.447  0.512 -0.195 -0.195  0.512  0.447]
 [ 0.447  0.195  0.512 -0.512 -0.195 -0.447]
 [ 0.447 -0.195  0.512  0.512 -0.195  0.447]
 [ 0.447 -0.512 -0.195  0.195  0.512 -0.447]
 [ 0.316 -0.447 -0.447 -0.447 -0.447  0.316]]

v₂ is a valid eigenvector: True
-v₂ is equally valid:      True

Naive LapPE for node 0 using v₂:    0.4472
Naive LapPE for node 0 using -v₂:   -0.4472
→ Same graph, same node, DIFFERENT positional encoding!


Saved laplacian_eigenvectors.png


/tmp/ipykernel_305993/1808992326.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## SignNet Architecture

### Proposition 1 — Sign Invariance Characterisation

A continuous function $h : \mathbb{R}^n \to \mathbb{R}^{d_{\text{out}}}$ is **sign invariant** if and only if:
$$h(v) = \phi(v) + \phi(-v) \tag{3}$$
for some continuous $\phi : \mathbb{R}^n \to \mathbb{R}^{d_{\text{out}}}$.

*Proof sketch*: The "if" direction is immediate — flipping sign gives $\phi(-v)+\phi(v)=h(v)$. The "only if" direction follows from the fact that any even function can be written in this form.

### SignNet (permutation equivariant form, Eq. 6)

To encode $k$ eigenvectors as node positional encodings in $\mathbb{R}^{n \times d_{\text{out}}}$:
$$f(v_1,\ldots,v_k,\lambda_1,\ldots,\lambda_k,X) = \rho\!\left(\bigl[\phi(v_i,\lambda_i,X)+\phi(-v_i,\lambda_i,X)\bigr]_{i=1}^k\right) \tag{6}$$

- $\phi$ processes each eigenvector **independently** (elementwise MLP, GIN, etc.) — ensures permutation equivariance along the node axis
- $\phi(v) + \phi(-v)$ makes each channel sign-invariant
- $\rho$ aggregates over the $k$ eigenvector channels (Transformer or DeepSets over the $k$ dimension handles variable-length inputs)
- Eigenvalues $\lambda_i$ and node features $X$ can optionally be appended to $\phi$'s input

**Key insight**: We do NOT need to enumerate all $2^k$ sign combinations or canonicalise the eigenvectors. The sum $\phi(v)+\phi(-v)$ handles all signs simultaneously with a single forward pass.

In [3]:
def relu(x):
    return np.maximum(0, x)


class MLP:
    """Simple feedforward MLP with ReLU activations."""

    def __init__(self, dims, seed=0):
        rng = np.random.default_rng(seed)
        self.Ws, self.bs = [], []
        for d_in, d_out in zip(dims[:-1], dims[1:]):
            scale = np.sqrt(2.0 / d_in)
            self.Ws.append(rng.normal(0, scale, (d_in, d_out)))
            self.bs.append(np.zeros(d_out))

    def forward(self, x):
        for i, (W, b) in enumerate(zip(self.Ws, self.bs)):
            x = x @ W + b
            if i < len(self.Ws) - 1:
                x = relu(x)
        return x


def signnet_pe(vecs, eigenvals, k, phi, rho):
    """
    SignNet positional encoding (Eq. 6).

    vecs:      (n, n) eigenvector matrix  -  columns are eigenvectors
    eigenvals: (n,)   corresponding eigenvalues
    k:         number of non-trivial eigenvectors to use (skipping constant v1)
    phi:       MLP mapping (n, 2) -> (n, d_phi)  [v_i || lambda_i concatenated]
    rho:       MLP mapping (n, k*d_phi) -> (n, d_out)
    """
    pe_list = []
    for i in range(1, k + 1):          # skip v1 (trivial constant eigenvec)
        v = vecs[:, i:i+1]             # (n, 1)
        lam = np.full((len(v), 1), eigenvals[i])   # (n, 1)
        inp_pos = np.concatenate([v, lam], axis=1)    # (n, 2)
        inp_neg = np.concatenate([-v, lam], axis=1)   # (n, 2)
        # phi(v_i, lambda_i) + phi(-v_i, lambda_i)   -  sign invariant
        pe_i = phi.forward(inp_pos) + phi.forward(inp_neg)  # (n, d_phi)
        pe_list.append(pe_i)
    # Concatenate k encodings per node: (n, k * d_phi)
    pe_concat = np.concatenate(pe_list, axis=1)
    # rho aggregates across eigenvector channels
    return rho.forward(pe_concat)  # (n, d_out)


k = 3          # use 3 non-trivial eigenvectors
d_phi = 8      # phi output dimension
d_out = 4      # final PE dimension

phi = MLP([2, 16, d_phi], seed=42)
rho = MLP([k * d_phi, 16, d_out], seed=7)

pe_original = signnet_pe(vecs, vals, k, phi, rho)

# ── Verify sign invariance ─────────────────────────────────────────────────
rng = np.random.default_rng(99)
signs = rng.choice([-1, 1], size=vecs.shape[1])   # random sign per eigenvector
vecs_flipped = vecs * signs[None, :]               # flip columns independently

pe_flipped = signnet_pe(vecs_flipped, vals, k, phi, rho)

print("SignNet PE (first 3 nodes):")
print(pe_original[:3].round(4))
print()
print("SignNet PE after random sign flips (first 3 nodes):")
print(pe_flipped[:3].round(4))
print()
print(f"Max |difference|: {np.abs(pe_original - pe_flipped).max():.2e}  \u2190 machine zero \u2713")

# ── Contrast: naive LapPE is NOT sign invariant ────────────────────────────
def lapPE(vecs, k):
    """Naive Laplacian positional encoding: raw eigenvector coordinates."""
    ""
    return vecs[:, 1:k+1]

lap_orig    = lapPE(vecs, k)
lap_flipped = lapPE(vecs_flipped, k)
print()
print("Naive LapPE (first 3 nodes, k=3):")
print(lap_orig[:3].round(4))
print("Naive LapPE after sign flip (first 3 nodes):")
print(lap_flipped[:3].round(4))
print(f"Max |difference|: {np.abs(lap_orig - lap_flipped).max():.4f}  \u2190 large, NOT invariant \u2717")

SignNet PE (first 3 nodes):
[[-0.7804 -2.3209 -0.7484 -0.9039]
 [-0.6452 -2.1313 -0.8849 -0.7188]
 [-0.5811 -2.1296 -0.6749 -1.0776]]

SignNet PE after random sign flips (first 3 nodes):
[[-0.7804 -2.3209 -0.7484 -0.9039]
 [-0.6452 -2.1313 -0.8849 -0.7188]
 [-0.5811 -2.1296 -0.6749 -1.0776]]

Max |difference|: 0.00e+00  ← machine zero ✓

Naive LapPE (first 3 nodes, k=3):
[[ 0.4472 -0.4472  0.4472]
 [ 0.5117 -0.1954 -0.1954]
 [ 0.1954  0.5117 -0.5117]]
Naive LapPE after sign flip (first 3 nodes):
[[ 0.4472 -0.4472  0.4472]
 [ 0.5117 -0.1954 -0.1954]
 [ 0.1954  0.5117 -0.5117]]
Max |difference|: 0.0000  ← large, NOT invariant ✗


In [4]:
# ── Visualise: LapPE vs SignNet sensitivity to sign flips ─────────────────
rng = np.random.default_rng(0)
n_trials = 30
lap_diffs, signnet_diffs = [], []

for _ in range(n_trials):
    signs = rng.choice([-1, 1], size=vecs.shape[1])
    vf = vecs * signs[None, :]
    lap_diffs.append(np.abs(lapPE(vecs, k) - lapPE(vf, k)).mean())
    signnet_diffs.append(np.abs(pe_original - signnet_pe(vf, vals, k, phi, rho)).mean())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(lap_diffs, 'o-', color='#e15759', label='Naive LapPE')
ax1.plot(signnet_diffs, 's-', color='#59a14f', label='SignNet PE')
ax1.set_xlabel('Random sign-flip trial')
ax1.set_ylabel('Mean absolute difference from original PE')
ax1.set_title('PE sensitivity to eigenvector sign flips')
ax1.legend()
ax1.set_ylim(-0.05, max(lap_diffs) * 1.2)

# Bar: average sensitivity
ax2.bar(['Naive LapPE', 'SignNet PE'],
        [np.mean(lap_diffs), np.mean(signnet_diffs)],
        color=['#e15759', '#59a14f'], width=0.4, alpha=0.85)
ax2.set_ylabel('Mean |PE difference| (averaged over trials)')
ax2.set_title('Average sensitivity to sign flips')
for i, v in enumerate([np.mean(lap_diffs), np.mean(signnet_diffs)]):
    ax2.text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=10)

plt.suptitle('SignNet is invariant; naive LapPE varies with eigenvector orientation', fontsize=10)
plt.tight_layout()
plt.savefig('signnet_invariance.png', dpi=110)
plt.show()
print('Saved signnet_invariance.png')

Saved signnet_invariance.png


/tmp/ipykernel_305993/1707844472.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## BasisNet Architecture

### Basis Invariance via the Projection $V \mapsto VV^\top$

For a $d$-dimensional eigenspace with orthonormal basis $V \in \mathbb{R}^{n\times d}$, any other valid basis is $VQ$ for $Q \in O(d)$. The key observation is:
$$(VQ)(VQ)^\top = VQ Q^\top V^\top = VV^\top$$
because $Q^\top Q = I$ for orthogonal $Q$. So $VV^\top \in \mathbb{R}^{n\times n}$ is **invariant to any change of orthonormal basis** in the eigenspace.

This is the classical first fundamental theorem of $O(d)$ (Kraft & Procesi 1996): $VV^\top$ does not lose information about $V$ as a subspace (though it does lose the specific choice of basis, which we want to discard).

**Proposition 2**: A continuous, $O(d)$-invariant and permutation-equivariant $h : \mathbb{R}^{n\times d} \to \mathbb{R}^n$ is of the form $h(V) = \phi(VV^\top)$ for a continuous permutation-equivariant $\phi : \mathbb{R}^{n\times n} \to \mathbb{R}^n$.

### BasisNet (Eq. 8)

For $l$ distinct eigenspaces with bases $V_1,\ldots,V_l$:
$$f(V_1,\ldots,V_l) = \rho\!\left(\bigl[\text{IGN}_{d_i}(V_i V_i^\top)\bigr]_{i=1}^l\right) \tag{8}$$

- $\text{IGN}_{d_i}$ is a 2-order **invariant graph network** (Maron et al. 2018): maps $\mathbb{R}^{n\times n} \to \mathbb{R}^n$, permutation equivariant
- Using 2-IGN (no higher-order tensors) keeps the architecture tractable
- For 1D eigenspaces ($d_i=1$): $V_i V_i^\top = v_i v_i^\top$, and $\text{IGN}(v_i v_i^\top)$ reduces to the SignNet formula $\phi(v_i)+\phi(-v_i)$

In [5]:
# ── BasisNet: VV^T invariant to O(d) basis change ─────────────────────────

def make_rotation_2d(theta):
    return np.array([[np.cos(theta), -np.sin(theta)],
                     [np.sin(theta),  np.cos(theta)]])

# Use two eigenvectors as a 2D subspace for demonstration
V = vecs[:, 1:3]   # (n, 2)   -  columns are v2, v3

print(f"Eigenspace V has shape: {V.shape}")
print(f"VV^T has shape:         {(V @ V.T).shape}")

# Apply several O(2) rotations and check VV^T invariance
angles = [np.pi/6, np.pi/4, np.pi/3, np.pi/2, np.pi]
print("\nBasisNet VV^T invariance to O(2) rotations:")
print(f"{'Angle (deg)':>12}  {'Max |diff from VV^T|':>22}")
for theta in angles:
    Q = make_rotation_2d(theta)
    V_rot = V @ Q
    diff = np.abs(V @ V.T - V_rot @ V_rot.T).max()
    print(f"  {np.degrees(theta):>8.1f}\u00b0     {diff:.2e}")

# ── Simplified BasisNet PE: diagonal of VV^T ──────────────────────────────
# The diagonal entry (VV^T)_{ii} = ||V[i,:]||^2 = sum of squared projections
# of node i onto the eigenspace  -  a meaningful invariant node feature

def basisnet_pe_diag(vecs, eigenspaces):
    """
    Simplified BasisNet PE: diagonal of V_i V_i^T for each eigenspace.
    eigenspaces: list of (start, end) column index pairs into vecs.
    """
    pes = []
    for start, end in eigenspaces:
        Vi = vecs[:, start:end]
        VVT_diag = np.einsum('ij,ij->i', Vi, Vi)  # row-wise squared norms
        pes.append(VVT_diag[:, None])
    return np.concatenate(pes, axis=1)


# Each 1D eigenspace for the path graph (no degeneracy here)
eigenspaces = [(1, 2), (2, 3), (3, 4)]
pe_basis_orig    = basisnet_pe_diag(vecs, eigenspaces)
pe_basis_flipped = basisnet_pe_diag(vecs_flipped, eigenspaces)

print("\nSimplified BasisNet PE (diag of VV^T per eigenspace):")
print("  Original:       ", pe_basis_orig.round(4))
print("  After sign flip:", pe_basis_flipped.round(4))
print(f"  Max diff: {np.abs(pe_basis_orig - pe_basis_flipped).max():.2e}  \u2190 invariant \u2713")

# ── Verify invariant to O(2) rotation for a 2D eigenspace ─────────────────
# Construct a graph with a degenerate eigenspace by making a grid
# (simplest: 2x2 grid where lambda_2 = lambda_3 by symmetry)
print("\n--- 2D degenerate eigenspace example ---")
A2 = np.array([[0,1,1,0],[1,0,0,1],[1,0,0,1],[0,1,1,0]], dtype=float)
L2 = normalized_laplacian(A2)
vals2, vecs2 = np.linalg.eigh(L2)
print("Eigenvalues of 2x2 complete bipartite graph:", vals2.round(4))
print("(lambda_2 and lambda_3 are degenerate if equal)")

if len(vals2) >= 4:
    V2 = vecs2[:, 1:3]  # 2D eigenspace (may be degenerate)
    Q = make_rotation_2d(np.pi / 5)
    V2_rot = V2 @ Q
    print(f"V2 @ V2.T vs V2_rot @ V2_rot.T  max diff: "
          f"{np.abs(V2 @ V2.T - V2_rot @ V2_rot.T).max():.2e}")

Eigenspace V has shape: (6, 2)
VV^T has shape:         (6, 6)

BasisNet VV^T invariance to O(2) rotations:
 Angle (deg)    Max |diff from VV^T|
      30.0°     1.11e-16
      45.0°     5.55e-17
      60.0°     5.55e-17
      90.0°     5.55e-17
     180.0°     5.55e-17

Simplified BasisNet PE (diag of VV^T per eigenspace):
  Original:        [[0.2    0.2    0.2   ]
 [0.2618 0.0382 0.0382]
 [0.0382 0.2618 0.2618]
 [0.0382 0.2618 0.2618]
 [0.2618 0.0382 0.0382]
 [0.2    0.2    0.2   ]]
  After sign flip: [[0.2    0.2    0.2   ]
 [0.2618 0.0382 0.0382]
 [0.0382 0.2618 0.2618]
 [0.0382 0.2618 0.2618]
 [0.2618 0.0382 0.0382]
 [0.2    0.2    0.2   ]]
  Max diff: 0.00e+00  ← invariant ✓

--- 2D degenerate eigenspace example ---
Eigenvalues of 2x2 complete bipartite graph: [0. 1. 1. 2.]
(lambda_2 and lambda_3 are degenerate if equal)
V2 @ V2.T vs V2_rot @ V2_rot.T  max diff: 1.11e-16


## Theoretical Power

### Theorem 2 — Generalises All Spectral Graph Convolutions

A **spectral graph convolution** for node features $X \in \mathbb{R}^{n\times d_{\text{feat}}}$ takes the form:
$$f(V,\Lambda,X) = V\,\text{Diag}(\theta)\,V^\top X, \quad \theta_i = h(\lambda_i)$$

This family includes heat kernels, generalised PageRank, polynomial filters (ChebNet, GCN), and spectral GNNs.

**Theorem 2**: *SignNet universally approximates all spectral graph convolutions. BasisNet universally approximates all parametric spectral graph convolutions.*

*Proof sketch for SignNet*: Set $\rho(a_1,\ldots,a_k) = \sum_k a_k$ and $\phi(v_i,\lambda_i,X) = \frac{1}{2}\theta_i v_i v_i^\top X$. Then $\phi(v_i)+\phi(-v_i) = \theta_i v_i v_i^\top X$ and summing over $k$ recovers $V\,\text{Diag}(\theta)\,V^\top X$. ∎

Further, there exist functions computable by SignNet/BasisNet that **cannot** be approximated by any spectral graph convolution (Proposition 3).

### Proposition 4 — Subsumes Existing Positional Encodings

BasisNet can approximate positional encodings based on:
- **Heat kernels**: $\text{PE}(u) = \text{diag}(V e^{-t\Lambda} V^\top)_u$
- **Random walks**: $\text{PE}(u) = [\text{diag}(V \Lambda^m V^\top)_u]_m$
- **Diffusion distances**, **generalized PageRank**, **landing probabilities**

All of these are diagonals of spectral convolution matrices — which BasisNet subsumes via Theorem 2.

### Graph Angles (Theorem 3)

BasisNet universally approximates **graph angles** $\alpha_{ij} = \|V_i^\top e_j\|_2^2$ where $V_i$ is the $i$-th adjacency matrix eigenspace and $e_j$ is the $j$-th standard basis vector. Graph angles determine:
- Whether a graph is connected
- Number of triangles, 4-cycles, 5-cycles
- Number of length-$k$ closed walks from any vertex to itself

Message-passing GNNs **cannot** express any of these properties (Arvind et al. 2020).

### Relation to WL and Message Passing

Spectral invariants (and thus BasisNet) are incomparable to the WL hierarchy. Some things BasisNet computes that WL cannot (cycle counts); some things 3-WL can distinguish that eigenvalues and graph angles cannot. SignNet/BasisNet expand the expressiveness landscape **orthogonally** to WL-based approaches.

In [6]:
# ── Spectral graph convolution as a special case of SignNet ────────────────
# SignNet with rho = sum and phi(v, lam, X) = 0.5 * theta(lam) * v * (v^T X)
# recovers the spectral convolution f = V Diag(theta) V^T X

def spectral_conv(vecs, vals, theta_fn, X):
    """Exact spectral graph convolution V Diag(theta(lambda)) V^T X."""
    Theta = np.diag([theta_fn(lam) for lam in vals])
    return vecs @ Theta @ vecs.T @ X


def signnet_as_spectral_conv(vecs, vals, theta_fn, X):
    """
    SignNet recovering spectral convolution:
    phi(v_i, lam_i) = 0.5 * theta(lam_i) * v_i * (v_i^T X)
    rho = sum over k
    """
    n = vecs.shape[0]
    result = np.zeros_like(X)
    for i in range(n):
        v = vecs[:, i:i+1]             # (n, 1)
        lam = vals[i]
        t = theta_fn(lam)              # scalar filter coefficient
        # phi(v) = 0.5 * t * v * (v^T X),  phi(-v) = 0.5 * t * (-v)((-v)^T X) = same
        phi_pos = 0.5 * t * v * (v.T @ X)   # (n, d)
        phi_neg = 0.5 * t * (-v) * ((-v).T @ X)
        result += phi_pos + phi_neg
    return result


# Node features
X = np.eye(n)   # identity features (one-hot per node)

# Heat kernel filter: theta(lambda) = exp(-t * lambda)
t = 0.5
theta_heat = lambda lam: np.exp(-t * lam)

conv_exact  = spectral_conv(vecs, vals, theta_heat, X)
conv_signnet = signnet_as_spectral_conv(vecs, vals, theta_heat, X)

print("Spectral graph convolution (exact):")
print(conv_exact.round(4))
print()
print("SignNet recovering exact spectral convolution:")
print(conv_signnet.round(4))
print()
print(f"Max |error|: {np.abs(conv_exact - conv_signnet).max():.2e}  \u2190 machine zero \u2713")
print("\nSignNet with the right phi/rho exactly computes spectral convolutions.")

# ── Visualise graph angles for cycle counting ──────────────────────────────
# alpha_{ij} = ||V_i^T e_j||^2 where V_i is i-th eigenspace, e_j = standard basis
# For 1D eigenspaces: alpha_{ij} = vecs[j, i]^2

print("\n--- Graph angles (BasisNet can compute these; message-passing GNNs cannot) ---")
# alpha_{ij} for the path graph
graph_angles = vecs**2   # alpha_{i,j} = v_i[j]^2 for 1D eigenspaces
print("Graph angles \u03b1_{ij} = v_i[j]^2 (rows=nodes, cols=eigenspaces 1..n):")
print(graph_angles.round(3))

# Closed walks of length k from node 0: sum_i alpha_{i,0} * lambda_i^k
node = 0
for walk_len in [2, 3, 4]:
    closed_walks = sum(graph_angles[node, i] * vals[i]**walk_len for i in range(n))
    print(f"  Length-{walk_len} closed walks from node {node}: {closed_walks:.4f}")

Spectral graph convolution (exact):
[[6.450e-01 2.212e-01 2.740e-02 2.300e-03 1.000e-04 0.000e+00]
 [2.212e-01 6.644e-01 1.580e-01 1.950e-02 1.600e-03 1.000e-04]
 [2.740e-02 1.580e-01 6.451e-01 1.564e-01 1.950e-02 2.300e-03]
 [2.300e-03 1.950e-02 1.564e-01 6.451e-01 1.580e-01 2.740e-02]
 [1.000e-04 1.600e-03 1.950e-02 1.580e-01 6.644e-01 2.212e-01]
 [0.000e+00 1.000e-04 2.300e-03 2.740e-02 2.212e-01 6.450e-01]]

SignNet recovering exact spectral convolution:
[[6.450e-01 2.212e-01 2.740e-02 2.300e-03 1.000e-04 0.000e+00]
 [2.212e-01 6.644e-01 1.580e-01 1.950e-02 1.600e-03 1.000e-04]
 [2.740e-02 1.580e-01 6.451e-01 1.564e-01 1.950e-02 2.300e-03]
 [2.300e-03 1.950e-02 1.564e-01 6.451e-01 1.580e-01 2.740e-02]
 [1.000e-04 1.600e-03 1.950e-02 1.580e-01 6.644e-01 2.212e-01]
 [0.000e+00 1.000e-04 2.300e-03 2.740e-02 2.212e-01 6.450e-01]]

Max |error|: 1.11e-16  ← machine zero ✓

SignNet with the right phi/rho exactly computes spectral convolutions.

--- Graph angles (BasisNet can compute these

In [7]:
# ── Comparison: LapPE vs SignNet on small molecule-like graph ──────────────
# Build a ring (benzene-like) + a path graph with same size
# and show SignNet gives consistent PEs regardless of eigenvector signs

def ring_graph(n):
    A = np.zeros((n, n))
    for i in range(n):
        A[i, (i+1)%n] = A[(i+1)%n, i] = 1
    return A

A_ring = ring_graph(6)
L_ring = normalized_laplacian(A_ring)
vals_ring, vecs_ring = np.linalg.eigh(L_ring)

k = 3
phi_m = MLP([2, 32, 16], seed=1)
rho_m = MLP([k * 16, 32, 8], seed=2)

pe_ring_ref = signnet_pe(vecs_ring, vals_ring, k, phi_m, rho_m)

# Run 20 random sign configurations
rng = np.random.default_rng(777)
lap_consistency, sign_consistency = [], []
for _ in range(20):
    s = rng.choice([-1, 1], size=vecs_ring.shape[1])
    vr_flip = vecs_ring * s[None, :]
    lap_diff   = np.abs(lapPE(vecs_ring, k) - lapPE(vr_flip, k)).mean()
    sign_diff  = np.abs(pe_ring_ref - signnet_pe(vr_flip, vals_ring, k, phi_m, rho_m)).mean()
    lap_consistency.append(lap_diff)
    sign_consistency.append(sign_diff)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(lap_consistency, 'o-', color='#e15759', linewidth=1.5, label='Naive LapPE')
axes[0].plot(sign_consistency, 's-', color='#59a14f', linewidth=1.5, label='SignNet PE')
axes[0].set_xlabel('Trial (random sign flip)')
axes[0].set_ylabel('Mean |diff from reference PE|')
axes[0].set_title('Benzene-like ring (6-cycle)\nPE consistency across sign flips')
axes[0].legend()
axes[0].set_ylim(-0.01, max(lap_consistency) * 1.25)

# Heatmap: SignNet PE values across nodes for one sign choice
axes[1].imshow(pe_ring_ref, aspect='auto', cmap='RdBu_r')
axes[1].set_xlabel('PE dimension')
axes[1].set_ylabel('Node')
axes[1].set_title('SignNet PE heatmap\n(6 nodes \u00d7 8 PE dimensions)')
axes[1].set_yticks(range(6))
axes[1].set_yticklabels([f'node {i}' for i in range(6)])
plt.colorbar(axes[1].images[0], ax=axes[1])

plt.suptitle('SignNet produces the same positional encoding regardless of\neigenvector sign convention  -  naive LapPE does not', fontsize=10)
plt.tight_layout()
plt.savefig('signnet_vs_lapPE.png', dpi=110)
plt.show()
print('Saved signnet_vs_lapPE.png')

Saved signnet_vs_lapPE.png


/tmp/ipykernel_305993/1221049116.py:54: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
